In [ ]:
#1.1
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Define the neural network
class POSTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, context_size):
        super(POSTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden = nn.Linear(embedding_dim * (2 * context_size + 1), hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)
        self.embedding.weight.data.uniform_(-0.01, 0.01)

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = embedded.view(x.shape[0], -1)
        hidden = torch.tanh(self.hidden(embedded))
        output = self.output(hidden)
        return output


class TwitterPOSDataset(Dataset):
    def __init__(self, file_path, word_to_idx, tag_to_idx, context_size):
        self.data = []
        self.word_to_idx = word_to_idx
        self.tag_to_idx = tag_to_idx
        self.context_size = context_size

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        tokens = ['<s>'] * context_size
        tags = ['<s>'] * context_size
        for line in lines:
            if line.strip():
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    word, tag = parts
                    tokens.append(word)
                    tags.append(tag)
            else:
                tokens.extend(['</s>'] * context_size)
                tags.extend(['</s>'] * context_size)
                tokens.extend(['<s>'] * context_size)
                tags.extend(['<s>'] * context_size)

        tokens.extend(['</s>'] * context_size)
        tags.extend(['</s>'] * context_size)

        for idx in range(self.context_size, len(tokens) - self.context_size):
            context = [self.word_to_idx.get(tokens[idx + i], self.word_to_idx['UUUNKKK']) for i in range(-self.context_size, self.context_size + 1)]
            target = self.tag_to_idx.get(tags[idx], self.tag_to_idx['X'])


            if any(index >= len(self.word_to_idx) for index in context):
                continue

            self.data.append((context, target))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0]), self.data[idx][1]


def train_model(model, train_loader, dev_loader, criterion, optimizer, epochs):
    best_dev_accuracy = 0
    for epoch in range(epochs):
        model.train()
        for batch_contexts, batch_tags in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_contexts)
            loss = criterion(outputs, batch_tags)
            loss.backward()
            optimizer.step()

        dev_accuracy = evaluate_model(model, dev_loader)
        print(f'Epoch {epoch+1}, Dev Accuracy: {dev_accuracy:.2f}%')

        if dev_accuracy > best_dev_accuracy:
            best_dev_accuracy = dev_accuracy
            torch.save(model.state_dict(), 'best_model.pth')

    print(f'Best Dev Accuracy: {best_dev_accuracy:.2f}%')


def evaluate_model(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_contexts, batch_tags in data_loader:
            outputs = model(batch_contexts)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_tags.size(0)
            correct += (predicted == batch_tags).sum().item()
    return 100 * correct / total


def debug_and_check_indices(train_loader, dev_loader, devtest_loader, word_to_idx):
    max_index = len(word_to_idx) - 1
    for loader in [train_loader, dev_loader, devtest_loader]:
        for batch_contexts, _ in loader:
            if torch.max(batch_contexts) > max_index:
                print(f"Found index out of range: Max index {torch.max(batch_contexts).item()}, Allowed max index {max_index}")
                return False
    return True


def main():
    EMBEDDING_DIM = 50
    HIDDEN_DIM = 128
    BATCH_SIZE = 1
    EPOCHS = 5
    LEARNING_RATE = 0.02

    word_to_idx = {}
    with open("/content/twitter-embeddings.txt", 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            word = line.split()[0]
            word_to_idx[word] = idx
    word_to_idx['UUUNKKK'] = len(word_to_idx)
    word_to_idx['<s>'] = len(word_to_idx)
    word_to_idx['</s>'] = len(word_to_idx)

    tag_to_idx = {
        'N': 0, 'O': 1, 'S': 2, 'L': 3, '^': 4, 'Z': 5, 'M': 6, 'V': 7, 'A': 8, 'R': 9,
        '!': 10, 'D': 11, 'P': 12, '&': 13, 'T': 14, 'X': 15, 'Y': 16, '#': 17, '@': 18,
        '~': 19, 'U': 20, 'E': 21, '$': 22, ',': 23, 'G': 24, '<s>': 25, '</s>': 26
    }

    for context_size in [0, 1]:
        print(f"Training with context size {context_size}")

        train_dataset = TwitterPOSDataset("/content/twpos-train.tsv", word_to_idx, tag_to_idx, context_size)
        dev_dataset = TwitterPOSDataset("/content/twpos-dev.tsv", word_to_idx, tag_to_idx, context_size)
        devtest_dataset = TwitterPOSDataset("/content/twpos-devtest.tsv", word_to_idx, tag_to_idx, context_size)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
        devtest_loader = DataLoader(devtest_dataset, batch_size=BATCH_SIZE)

        model = POSTagger(len(word_to_idx), EMBEDDING_DIM, HIDDEN_DIM, len(tag_to_idx), context_size)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)


        if not debug_and_check_indices(train_loader, dev_loader, devtest_loader, word_to_idx):
            print("Exiting due to index errors.")
            return

        train_model(model, train_loader, dev_loader, criterion, optimizer, EPOCHS)

        model.load_state_dict(torch.load('best_model.pth'))
        devtest_accuracy = evaluate_model(model, devtest_loader)
        print(f'Context size {context_size}, DevTest Accuracy: {devtest_accuracy:.2f}%')

if __name__ == '__main__':
    main()


Training with context size 0
Epoch 1, Dev Accuracy: 70.09%
Epoch 2, Dev Accuracy: 76.15%
Epoch 3, Dev Accuracy: 78.10%
Epoch 4, Dev Accuracy: 76.73%
Epoch 5, Dev Accuracy: 78.12%
Best Dev Accuracy: 78.12%


<ipython-input-3-6aa2a5439d8a>:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Context size 0, DevTest Accuracy: 79.24%
Training with context size 1
Epoch 1, Dev Accuracy: 73.25%
Epoch 2, Dev Accuracy: 80.17%
Epoch 3, Dev Accuracy: 81.29%
Epoch 4, Dev Accuracy: 80.42%
Epoch 5, Dev Accuracy: 79.80%
Best Dev Accuracy: 81.29%
Context size 1, DevTest Accuracy: 82.42%


In [ ]:
#1.2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re

# Define the neural network
class POSTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, context_size, num_features):
        super(POSTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden = nn.Linear(embedding_dim * (2 * context_size + 1) + num_features, hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)
        self.embedding.weight.data.uniform_(-0.01, 0.01)

    def forward(self, x, features):
        embedded = self.embedding(x)
        embedded = embedded.view(x.shape[0], -1)
        combined = torch.cat((embedded, features), dim=1)
        hidden = torch.tanh(self.hidden(combined))
        output = self.output(hidden)
        return output

# Feature extraction
def extract_features(word, prev_word, next_word):
    features = []
    features.append(int(word.istitle()))  # Is capitalized
    features.append(int(word.isupper()))  # Is all uppercase
    features.append(int(bool(re.search(r'\d', word))))  # Contains digit
    features.append(int('@' in word))  # Contains @
    features.append(int('#' in word))  # Contains #
    features.append(len(word))  # Word length
    features.append(int(word.startswith('un')))  # Starts with 'un'
    features.append(int(prev_word.istitle()))  # Previous word is capitalized
    features.append(int(next_word.istitle()))  # Next word is capitalized
    features.append(int(word.lower() in ['the', 'a', 'an']))  # Is article
    return features


class TwitterPOSDataset(Dataset):
    def __init__(self, file_path, word_to_idx, tag_to_idx, context_size):
        self.data = []
        self.word_to_idx = word_to_idx
        self.tag_to_idx = tag_to_idx
        self.context_size = context_size

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        tokens = ['<s>'] * context_size + ['<s>']
        tags = ['<s>'] * context_size + ['<s>']
        for line in lines:
            if line.strip():
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    word, tag = parts
                    tokens.append(word)
                    tags.append(tag)
            else:
                tokens.extend(['</s>'] * (context_size + 1))
                tags.extend(['</s>'] * (context_size + 1))
                tokens.extend(['<s>'] * (context_size + 1))
                tags.extend(['<s>'] * (context_size + 1))

        tokens.extend(['</s>'] * (context_size + 1))
        tags.extend(['</s>'] * (context_size + 1))

        for idx in range(context_size, len(tokens) - context_size - 1):
            context = [self.word_to_idx.get(tokens[idx + i], self.word_to_idx['UUUNKKK']) for i in range(-context_size, context_size + 1)]
            target = self.tag_to_idx.get(tags[idx], self.tag_to_idx['X'])


            features = extract_features(tokens[idx], tokens[idx-1], tokens[idx+1])


            if any(index >= len(self.word_to_idx) for index in context):
                continue

            self.data.append((context, target, features))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0]), self.data[idx][1], torch.tensor(self.data[idx][2], dtype=torch.float)


def train_model(model, train_loader, dev_loader, criterion, optimizer, epochs):
    best_dev_accuracy = 0
    for epoch in range(epochs):
        model.train()
        for batch_contexts, batch_tags, batch_features in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_contexts, batch_features)
            loss = criterion(outputs, batch_tags)
            loss.backward()
            optimizer.step()

        dev_accuracy = evaluate_model(model, dev_loader)
        print(f'Epoch {epoch+1}, Dev Accuracy: {dev_accuracy:.2f}%')

        if dev_accuracy > best_dev_accuracy:
            best_dev_accuracy = dev_accuracy
            torch.save(model.state_dict(), 'best_model.pth')

    print(f'Best Dev Accuracy: {best_dev_accuracy:.2f}%')


def evaluate_model(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_contexts, batch_tags, batch_features in data_loader:
            outputs = model(batch_contexts, batch_features)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_tags.size(0)
            correct += (predicted == batch_tags).sum().item()
    return 100 * correct / total


def debug_and_check_indices(train_loader, dev_loader, devtest_loader, word_to_idx):
    max_index = len(word_to_idx) - 1
    for loader in [train_loader, dev_loader, devtest_loader]:
        for batch_contexts, _, _ in loader:
            if torch.max(batch_contexts) > max_index:
                print(f"Found index out of range: Max index {torch.max(batch_contexts).item()}, Allowed max index {max_index}")
                return False
    return True


def main():
    EMBEDDING_DIM = 50
    HIDDEN_DIM = 128
    BATCH_SIZE = 1
    EPOCHS = 5
    LEARNING_RATE = 0.02
    NUM_FEATURES = 10  # Set to 10

    word_to_idx = {}
    with open("/content/twitter-embeddings.txt", 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            word = line.split()[0]
            word_to_idx[word] = idx
    word_to_idx['UUUNKKK'] = len(word_to_idx)
    word_to_idx['<s>'] = len(word_to_idx)
    word_to_idx['</s>'] = len(word_to_idx)

    tag_to_idx = {
        'N': 0, 'O': 1, 'S': 2, 'L': 3, '^': 4, 'Z': 5, 'M': 6, 'V': 7, 'A': 8, 'R': 9,
        '!': 10, 'D': 11, 'P': 12, '&': 13, 'T': 14, 'X': 15, 'Y': 16, '#': 17, '@': 18,
        '~': 19, 'U': 20, 'E': 21, '$': 22, ',': 23, 'G': 24, '<s>': 25, '</s>': 26
    }

    for context_size in [0, 1]:
        print(f"Training with context size {context_size}")

        train_dataset = TwitterPOSDataset("/content/twpos-train.tsv", word_to_idx, tag_to_idx, context_size)
        dev_dataset = TwitterPOSDataset("/content/twpos-dev.tsv", word_to_idx, tag_to_idx, context_size)
        devtest_dataset = TwitterPOSDataset("/content/twpos-devtest.tsv", word_to_idx, tag_to_idx, context_size)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
        devtest_loader = DataLoader(devtest_dataset, batch_size=BATCH_SIZE)

        model = POSTagger(len(word_to_idx), EMBEDDING_DIM, HIDDEN_DIM, len(tag_to_idx), context_size, NUM_FEATURES)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)


        if not debug_and_check_indices(train_loader, dev_loader, devtest_loader, word_to_idx):
            print("Exiting due to index errors.")
            return

        train_model(model, train_loader, dev_loader, criterion, optimizer, EPOCHS)

        model.load_state_dict(torch.load('best_model.pth'))
        devtest_accuracy = evaluate_model(model, devtest_loader)
        print(f'Context size {context_size}, DevTest Accuracy: {devtest_accuracy:.2f}%')

if __name__ == '__main__':
    main()


Training with context size 0
Epoch 1, Dev Accuracy: 63.86%
Epoch 2, Dev Accuracy: 73.99%
Epoch 3, Dev Accuracy: 79.84%
Epoch 4, Dev Accuracy: 80.05%
Epoch 5, Dev Accuracy: 77.14%
Best Dev Accuracy: 80.05%


<ipython-input-15-23b6538cf6bc>:188: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Context size 0, DevTest Accuracy: 81.20%
Training with context size 1
Epoch 1, Dev Accuracy: 73.55%
Epoch 2, Dev Accuracy: 80.48%
Epoch 3, Dev Accuracy: 82.58%
Epoch 4, Dev Accuracy: 83.03%
Epoch 5, Dev Accuracy: 83.49%
Best Dev Accuracy: 83.49%
Context size 1, DevTest Accuracy: 84.09%


In [ ]:
#1.3 without features


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


class POSTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, context_size):
        super(POSTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden = nn.Linear(embedding_dim * (2 * context_size + 1), hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = embedded.view(x.shape[0], -1)
        hidden = torch.tanh(self.hidden(embedded))
        output = self.output(hidden)
        return output


class TwitterPOSDataset(Dataset):
    def __init__(self, file_path, word_to_idx, tag_to_idx, context_size):
        self.data = []
        self.word_to_idx = word_to_idx
        self.tag_to_idx = tag_to_idx
        self.context_size = context_size

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        tokens = ['<s>'] * context_size
        tags = ['<s>'] * context_size
        for line in lines:
            if line.strip():
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    word, tag = parts
                    tokens.append(word)
                    tags.append(tag)
            else:
                tokens.extend(['</s>'] * context_size)
                tags.extend(['</s>'] * context_size)
                tokens.extend(['<s>'] * context_size)
                tags.extend(['<s>'] * context_size)

        tokens.extend(['</s>'] * context_size)
        tags.extend(['</s>'] * context_size)

        for idx in range(self.context_size, len(tokens) - self.context_size):
            context = [self.word_to_idx.get(tokens[idx + i], self.word_to_idx['UUUNKKK']) for i in range(-self.context_size, self.context_size + 1)]
            target = self.tag_to_idx.get(tags[idx], self.tag_to_idx['X'])
            self.data.append((context, target))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0]), self.data[idx][1]


def train_model(model, train_loader, dev_loader, criterion, optimizer, epochs):
    best_dev_accuracy = 0
    for epoch in range(epochs):
        model.train()
        for batch_contexts, batch_tags in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_contexts)
            loss = criterion(outputs, batch_tags)
            loss.backward()
            optimizer.step()

        dev_accuracy = evaluate_model(model, dev_loader)
        print(f'Epoch {epoch+1}, Dev Accuracy: {dev_accuracy:.2f}%')

        if dev_accuracy > best_dev_accuracy:
            best_dev_accuracy = dev_accuracy
            torch.save(model.state_dict(), 'best_model.pth')

    print(f'Best Dev Accuracy: {best_dev_accuracy:.2f}%')


def evaluate_model(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_contexts, batch_tags in data_loader:
            outputs = model(batch_contexts)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_tags.size(0)
            correct += (predicted == batch_tags).sum().item()
    return 100 * correct / total


def main():
    EMBEDDING_DIM = 50
    HIDDEN_DIM = 128
    BATCH_SIZE = 1
    EPOCHS = 10
    LEARNING_RATE = 0.02

    word_to_idx = {}
    embeddings = []
    with open('twitter-embeddings.txt', 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            parts = line.strip().split()
            word = parts[0]
            embedding = [float(val) for val in parts[1:]]
            word_to_idx[word] = idx
            embeddings.append(embedding)


    for token in ['UUUNKKK', '<s>']:
        if token not in word_to_idx:
            word_to_idx[token] = len(word_to_idx)
            embeddings.append([0.0] * EMBEDDING_DIM)  # Initialize with zeros


    word_to_idx['<s>'] = word_to_idx['</s>']

    tag_to_idx = {
        'N': 0, 'O': 1, 'S': 2, 'L': 3, '^': 4, 'Z': 5, 'M': 6, 'V': 7, 'A': 8, 'R': 9,
        '!': 10, 'D': 11, 'P': 12, '&': 13, 'T': 14, 'X': 15, 'Y': 16, '#': 17, '@': 18,
        '~': 19, 'U': 20, 'E': 21, '$': 22, ',': 23, 'G': 24, '<s>': 25, '</s>': 26
    }

    for context_size in [0, 1]:
        print(f"Training with context size {context_size}")

        train_dataset = TwitterPOSDataset('twpos-train.tsv', word_to_idx, tag_to_idx, context_size)
        dev_dataset = TwitterPOSDataset('twpos-dev.tsv', word_to_idx, tag_to_idx, context_size)
        devtest_dataset = TwitterPOSDataset('twpos-devtest.tsv', word_to_idx, tag_to_idx, context_size)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
        devtest_loader = DataLoader(devtest_dataset, batch_size=BATCH_SIZE)

        model = POSTagger(len(word_to_idx), EMBEDDING_DIM, HIDDEN_DIM, len(tag_to_idx), context_size)


        model.embedding.weight.data.copy_(torch.tensor(embeddings))

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

        train_model(model, train_loader, dev_loader, criterion, optimizer, EPOCHS)

        model.load_state_dict(torch.load('best_model.pth'))
        devtest_accuracy = evaluate_model(model, devtest_loader)
        print(f'Context size {context_size}, DevTest Accuracy: {devtest_accuracy:.2f}%')


        if context_size == 1:
            print("Training with fixed embeddings")
            model_fixed = POSTagger(len(word_to_idx), EMBEDDING_DIM, HIDDEN_DIM, len(tag_to_idx), context_size)
            model_fixed.embedding.weight.data.copy_(torch.tensor(embeddings))
            model_fixed.embedding.weight.requires_grad = False

            optimizer_fixed = optim.SGD(filter(lambda p: p.requires_grad, model_fixed.parameters()), lr=LEARNING_RATE)

            train_model(model_fixed, train_loader, dev_loader, criterion, optimizer_fixed, EPOCHS)

            model_fixed.load_state_dict(torch.load('best_model.pth'))
            devtest_accuracy_fixed = evaluate_model(model_fixed, devtest_loader)
            print(f'Fixed embeddings, DevTest Accuracy: {devtest_accuracy_fixed:.2f}%')

if __name__ == '__main__':
    main()


Training with context size 0
Epoch 1, Dev Accuracy: 84.07%
Epoch 2, Dev Accuracy: 82.78%
Epoch 3, Dev Accuracy: 82.14%
Epoch 4, Dev Accuracy: 84.57%
Epoch 5, Dev Accuracy: 84.38%
Epoch 6, Dev Accuracy: 84.67%
Epoch 7, Dev Accuracy: 82.99%
Epoch 8, Dev Accuracy: 83.95%
Epoch 9, Dev Accuracy: 82.68%
Epoch 10, Dev Accuracy: 82.51%
Best Dev Accuracy: 84.67%


<ipython-input-1-b6255590b4ae>:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Context size 0, DevTest Accuracy: 84.44%
Training with context size 1
Epoch 1, Dev Accuracy: 87.42%
Epoch 2, Dev Accuracy: 88.75%
Epoch 3, Dev Accuracy: 88.42%
Epoch 4, Dev Accuracy: 88.62%
Epoch 5, Dev Accuracy: 87.51%
Epoch 6, Dev Accuracy: 88.02%
Epoch 7, Dev Accuracy: 88.27%
Epoch 8, Dev Accuracy: 88.15%
Epoch 9, Dev Accuracy: 87.67%
Epoch 10, Dev Accuracy: 86.67%
Best Dev Accuracy: 88.75%
Context size 1, DevTest Accuracy: 88.83%
Training with fixed embeddings
Epoch 1, Dev Accuracy: 86.08%
Epoch 2, Dev Accuracy: 86.48%
Epoch 3, Dev Accuracy: 87.49%
Epoch 4, Dev Accuracy: 87.43%
Epoch 5, Dev Accuracy: 87.27%
Epoch 6, Dev Accuracy: 87.31%
Epoch 7, Dev Accuracy: 87.91%
Epoch 8, Dev Accuracy: 87.62%
Epoch 9, Dev Accuracy: 88.09%
Epoch 10, Dev Accuracy: 88.16%
Best Dev Accuracy: 88.16%


<ipython-input-1-b6255590b4ae>:168: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_fixed.load_state_dict(torch.load('best_model.pth'))


Fixed embeddings, DevTest Accuracy: 87.47%


In [2]:
#1.3 with features-
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re


class POSTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, context_size, num_features):
        super(POSTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden = nn.Linear(embedding_dim * (2 * context_size + 1) + num_features, hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, features):
        embedded = self.embedding(x)
        embedded = embedded.view(x.shape[0], -1)
        combined = torch.cat((embedded, features), dim=1)
        hidden = torch.tanh(self.hidden(combined))
        output = self.output(hidden)
        return output


def extract_features(word, prev_word, next_word):
    features = []
    features.append(int(word.istitle()))  # Is capitalized
    features.append(int(word.isupper()))  # Is all uppercase
    features.append(int(bool(re.search(r'\d', word))))  # Contains digit
    features.append(int('@' in word))  # Contains @
    features.append(int('#' in word))  # Contains #
    features.append(len(word))  # Word length
    features.append(int(word.startswith('un')))  # Starts with 'un'
    features.append(int(prev_word.istitle()))  # Previous word is capitalized
    features.append(int(next_word.istitle()))  # Next word is capitalized
    features.append(int(word.lower() in ['the', 'a', 'an']))  # Is article
    return features


class TwitterPOSDataset(Dataset):
    def __init__(self, file_path, word_to_idx, tag_to_idx, context_size):
        self.data = []
        self.word_to_idx = word_to_idx
        self.tag_to_idx = tag_to_idx
        self.context_size = context_size

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        tokens = ['<s>'] * context_size + ['<s>']
        tags = ['<s>'] * context_size + ['<s>']
        for line in lines:
            if line.strip():
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    word, tag = parts
                    tokens.append(word)
                    tags.append(tag)
            else:
                tokens.extend(['</s>'] * (context_size + 1))
                tags.extend(['</s>'] * (context_size + 1))
                tokens.extend(['<s>'] * (context_size + 1))
                tags.extend(['<s>'] * (context_size + 1))

        tokens.extend(['</s>'] * (context_size + 1))
        tags.extend(['</s>'] * (context_size + 1))

        for idx in range(context_size, len(tokens) - context_size - 1):
            context = [self.word_to_idx.get(tokens[idx + i], self.word_to_idx['UUUNKKK']) for i in range(-context_size, context_size + 1)]
            target = self.tag_to_idx.get(tags[idx], self.tag_to_idx['X'])


            features = extract_features(tokens[idx], tokens[idx-1], tokens[idx+1])

            self.data.append((context, target, features))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0]), self.data[idx][1], torch.tensor(self.data[idx][2], dtype=torch.float)


def train_model(model, train_loader, dev_loader, criterion, optimizer, epochs):
    best_dev_accuracy = 0
    for epoch in range(epochs):
        model.train()
        for batch_contexts, batch_tags, batch_features in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_contexts, batch_features)
            loss = criterion(outputs, batch_tags)
            loss.backward()
            optimizer.step()

        dev_accuracy = evaluate_model(model, dev_loader)
        print(f'Epoch {epoch+1}, Dev Accuracy: {dev_accuracy:.2f}%')

        if dev_accuracy > best_dev_accuracy:
            best_dev_accuracy = dev_accuracy
            torch.save(model.state_dict(), 'best_model.pth')

    print(f'Best Dev Accuracy: {best_dev_accuracy:.2f}%')


def evaluate_model(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_contexts, batch_tags, batch_features in data_loader:
            outputs = model(batch_contexts, batch_features)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_tags.size(0)
            correct += (predicted == batch_tags).sum().item()
    return 100 * correct / total


def main():
    EMBEDDING_DIM = 50
    HIDDEN_DIM = 128
    BATCH_SIZE = 1
    EPOCHS = 5
    LEARNING_RATE = 0.02
    NUM_FEATURES = 10  # Set to 10

    # Load pre-trained word embeddings
    word_to_idx = {}
    embeddings = []
    with open('twitter-embeddings.txt', 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            parts = line.strip().split()
            word = parts[0]
            embedding = [float(val) for val in parts[1:]]
            word_to_idx[word] = idx
            embeddings.append(embedding)


    for token in ['UUUNKKK', '<s>']:
        if token not in word_to_idx:
            word_to_idx[token] = len(word_to_idx)
            embeddings.append([0.0] * EMBEDDING_DIM)  # Initialize with zeros


    word_to_idx['<s>'] = word_to_idx['</s>']

    tag_to_idx = {
        'N': 0, 'O': 1, 'S': 2, 'L': 3, '^': 4, 'Z': 5, 'M': 6, 'V': 7, 'A': 8, 'R': 9,
        '!': 10, 'D': 11, 'P': 12, '&': 13, 'T': 14, 'X': 15, 'Y': 16, '#': 17, '@': 18,
        '~': 19, 'U': 20, 'E': 21, '$': 22, ',': 23, 'G': 24, '<s>': 25, '</s>': 26
    }

    for context_size in [0, 1]:
        print(f"Training with context size {context_size}")

        train_dataset = TwitterPOSDataset('twpos-train.tsv', word_to_idx, tag_to_idx, context_size)
        dev_dataset = TwitterPOSDataset('twpos-dev.tsv', word_to_idx, tag_to_idx, context_size)
        devtest_dataset = TwitterPOSDataset('twpos-devtest.tsv', word_to_idx, tag_to_idx, context_size)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
        devtest_loader = DataLoader(devtest_dataset, batch_size=BATCH_SIZE)

        model = POSTagger(len(word_to_idx), EMBEDDING_DIM, HIDDEN_DIM, len(tag_to_idx), context_size, NUM_FEATURES)


        model.embedding.weight.data.copy_(torch.tensor(embeddings))

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

        train_model(model, train_loader, dev_loader, criterion, optimizer, EPOCHS)

        model.load_state_dict(torch.load('best_model.pth'))
        devtest_accuracy = evaluate_model(model, devtest_loader)
        print(f'Context size {context_size}, DevTest Accuracy: {devtest_accuracy:.2f}%')


        if context_size == 1:
            print("Training with fixed embeddings")
            model_fixed = POSTagger(len(word_to_idx), EMBEDDING_DIM, HIDDEN_DIM, len(tag_to_idx), context_size, NUM_FEATURES)
            model_fixed.embedding.weight.data.copy_(torch.tensor(embeddings))
            model_fixed.embedding.weight.requires_grad = False

            optimizer_fixed = optim.SGD(filter(lambda p: p.requires_grad, model_fixed.parameters()), lr=LEARNING_RATE)

            train_model(model_fixed, train_loader, dev_loader, criterion, optimizer_fixed, EPOCHS)

            model_fixed.load_state_dict(torch.load('best_model.pth'))
            devtest_accuracy_fixed = evaluate_model(model_fixed, devtest_loader)
            print(f'Fixed embeddings, DevTest Accuracy: {devtest_accuracy_fixed:.2f}%')

if __name__ == '__main__':
    main()


Training with context size 0
Epoch 1, Dev Accuracy: 84.62%
Epoch 2, Dev Accuracy: 79.51%
Epoch 3, Dev Accuracy: 85.98%
Epoch 4, Dev Accuracy: 86.30%
Epoch 5, Dev Accuracy: 85.87%
Best Dev Accuracy: 86.30%


<ipython-input-2-1810a99e7776>:173: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Context size 0, DevTest Accuracy: 86.14%
Training with context size 1
Epoch 1, Dev Accuracy: 87.31%
Epoch 2, Dev Accuracy: 89.61%
Epoch 3, Dev Accuracy: 89.71%
Epoch 4, Dev Accuracy: 90.18%
Epoch 5, Dev Accuracy: 89.76%
Best Dev Accuracy: 90.18%
Context size 1, DevTest Accuracy: 89.90%
Training with fixed embeddings
Epoch 1, Dev Accuracy: 86.93%
Epoch 2, Dev Accuracy: 85.22%
Epoch 3, Dev Accuracy: 88.92%
Epoch 4, Dev Accuracy: 86.33%
Epoch 5, Dev Accuracy: 87.90%
Best Dev Accuracy: 88.92%


<ipython-input-2-1810a99e7776>:188: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_fixed.load_state_dict(torch.load('best_model.pth'))


Fixed embeddings, DevTest Accuracy: 88.79%


In [ ]:
#1.5

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Define the neural network models
class RNNTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, rnn_type='rnn', bidirectional=False):
        super(RNNTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.bidirectional = bidirectional

        if rnn_type == 'rnn':
            self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True, bidirectional=bidirectional)
        elif rnn_type == 'lstm':
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=bidirectional)
        elif rnn_type == 'gru':
            self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=bidirectional)

        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        return self.fc(output)


class TwitterPOSDataset(Dataset):
    def __init__(self, file_path, word_to_idx, tag_to_idx, sequence_length):
        self.data = []
        self.word_to_idx = word_to_idx
        self.tag_to_idx = tag_to_idx
        self.sequence_length = sequence_length

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        current_sequence = []
        current_tags = []
        for line in lines:
            if line.strip():
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    word, tag = parts
                    current_sequence.append(self.word_to_idx.get(word, self.word_to_idx['UUUNKKK']))
                    current_tags.append(self.tag_to_idx.get(tag, self.tag_to_idx['X']))
            else:
                if current_sequence:
                    self.data.append((current_sequence, current_tags))
                current_sequence = []
                current_tags = []

        if current_sequence:
            self.data.append((current_sequence, current_tags))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sequence, tags = self.data[idx]
        if len(sequence) > self.sequence_length:
            sequence = sequence[:self.sequence_length]
            tags = tags[:self.sequence_length]
        else:
            padding = [self.word_to_idx['<pad>']] * (self.sequence_length - len(sequence))
            sequence.extend(padding)
            tags.extend([self.tag_to_idx['<pad>']] * (self.sequence_length - len(tags)))
        return torch.tensor(sequence), torch.tensor(tags)


def train_model(model, train_loader, dev_loader, criterion, optimizer, epochs):
    best_dev_accuracy = 0
    for epoch in range(epochs):
        model.train()
        for batch_sequences, batch_tags in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_sequences)
            loss = criterion(outputs.view(-1, outputs.shape[-1]), batch_tags.view(-1))
            loss.backward()
            optimizer.step()

        dev_accuracy = evaluate_model(model, dev_loader)
        print(f'Epoch {epoch+1}, Dev Accuracy: {dev_accuracy:.2f}%')

        if dev_accuracy > best_dev_accuracy:
            best_dev_accuracy = dev_accuracy
            torch.save(model.state_dict(), 'best_model.pth')

    print(f'Best Dev Accuracy: {best_dev_accuracy:.2f}%')


def evaluate_model(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_sequences, batch_tags in data_loader:
            outputs = model(batch_sequences)
            _, predicted = torch.max(outputs, 2)
            mask = (batch_tags != data_loader.dataset.tag_to_idx['<pad>'])
            correct += ((predicted == batch_tags) * mask).sum().item()
            total += mask.sum().item()
    return 100 * correct / total


def main():
    EMBEDDING_DIM = 50
    HIDDEN_DIM = 128
    BATCH_SIZE = 32
    EPOCHS = 10
    LEARNING_RATE = 0.01
    SEQUENCE_LENGTH = 50

    word_to_idx = {}
    with open("/content/twitter-embeddings.txt", 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            word = line.split()[0]
            word_to_idx[word] = idx
    word_to_idx['UUUNKKK'] = len(word_to_idx)
    word_to_idx['<pad>'] = len(word_to_idx)

    tag_to_idx = {
        'N': 0, 'O': 1, 'S': 2, 'L': 3, '^': 4, 'Z': 5, 'M': 6, 'V': 7, 'A': 8, 'R': 9,
        '!': 10, 'D': 11, 'P': 12, '&': 13, 'T': 14, 'X': 15, 'Y': 16, '#': 17, '@': 18,
        '~': 19, 'U': 20, 'E': 21, '$': 22, ',': 23, 'G': 24, '<pad>': 25
    }

    models = [
        {'type': 'rnn', 'bidirectional': False},
        {'type': 'lstm', 'bidirectional': False},
        {'type': 'gru', 'bidirectional': False},
        {'type': 'rnn', 'bidirectional': True},
    ]

    train_dataset = TwitterPOSDataset("/content/twpos-train.tsv", word_to_idx, tag_to_idx, SEQUENCE_LENGTH)
    dev_dataset = TwitterPOSDataset("/content/twpos-dev.tsv", word_to_idx, tag_to_idx, SEQUENCE_LENGTH)
    devtest_dataset = TwitterPOSDataset("/content/twpos-devtest.tsv", word_to_idx, tag_to_idx, SEQUENCE_LENGTH)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
    devtest_loader = DataLoader(devtest_dataset, batch_size=BATCH_SIZE)

    for model_config in models:
        print(f"\nTraining {model_config['type'].upper()} {'(Bidirectional)' if model_config['bidirectional'] else ''}")
        model = RNNTagger(len(word_to_idx), EMBEDDING_DIM, HIDDEN_DIM, len(tag_to_idx),
                          rnn_type=model_config['type'], bidirectional=model_config['bidirectional'])

        criterion = nn.CrossEntropyLoss(ignore_index=tag_to_idx['<pad>'])
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        train_model(model, train_loader, dev_loader, criterion, optimizer, EPOCHS)

        model.load_state_dict(torch.load('best_model.pth'))
        devtest_accuracy = evaluate_model(model, devtest_loader)
        print(f'DevTest Accuracy: {devtest_accuracy:.2f}%')

if __name__ == '__main__':
    main()



Training RNN 
Epoch 1, Dev Accuracy: 62.64%
Epoch 2, Dev Accuracy: 70.30%
Epoch 3, Dev Accuracy: 73.24%
Epoch 4, Dev Accuracy: 74.82%
Epoch 5, Dev Accuracy: 75.77%
Epoch 6, Dev Accuracy: 75.67%
Epoch 7, Dev Accuracy: 76.39%
Epoch 8, Dev Accuracy: 75.77%
Epoch 9, Dev Accuracy: 76.21%
Epoch 10, Dev Accuracy: 76.10%
Best Dev Accuracy: 76.39%
DevTest Accuracy: 77.28%

Training LSTM 


<ipython-input-6-bcd5a098c2dc>:155: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Epoch 1, Dev Accuracy: 64.70%
Epoch 2, Dev Accuracy: 73.24%
Epoch 3, Dev Accuracy: 76.04%
Epoch 4, Dev Accuracy: 76.60%
Epoch 5, Dev Accuracy: 77.31%
Epoch 6, Dev Accuracy: 77.14%
Epoch 7, Dev Accuracy: 77.54%
Epoch 8, Dev Accuracy: 77.04%
Epoch 9, Dev Accuracy: 76.87%
Epoch 10, Dev Accuracy: 77.14%
Best Dev Accuracy: 77.54%
DevTest Accuracy: 77.93%

Training GRU 
Epoch 1, Dev Accuracy: 66.29%
Epoch 2, Dev Accuracy: 73.22%
Epoch 3, Dev Accuracy: 75.54%
Epoch 4, Dev Accuracy: 76.06%
Epoch 5, Dev Accuracy: 77.04%
Epoch 6, Dev Accuracy: 76.69%
Epoch 7, Dev Accuracy: 76.64%
Epoch 8, Dev Accuracy: 76.52%
Epoch 9, Dev Accuracy: 76.56%
Epoch 10, Dev Accuracy: 76.62%
Best Dev Accuracy: 77.04%
DevTest Accuracy: 77.75%

Training RNN (Bidirectional)
Epoch 1, Dev Accuracy: 66.29%
Epoch 2, Dev Accuracy: 71.98%
Epoch 3, Dev Accuracy: 75.01%
Epoch 4, Dev Accuracy: 76.29%
Epoch 5, Dev Accuracy: 76.25%
Epoch 6, Dev Accuracy: 76.52%
Epoch 7, Dev Accuracy: 77.31%
Epoch 8, Dev Accuracy: 77.52%
Epoch 9, De